# Lab 06-09 · Maximum Power Transfer

Companion to chapter [06-09](../../chapters/06-09-maximum-power-transfer.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import numpy as np
from scipy.optimize import minimize_scalar
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — Sweeping the load in ngspice

[maxpower.cir](maxpower.cir) represents each load by a swept voltage source (the load voltage) and finds the peak of $p = v\,i$.

**🔮 Predict:** for Example 06-09.1 (10 V, 8 Ω), what is the peak power, and how much lower is it with the load a factor of 2 off?

In [2]:
if HAVE_NGSPICE:
    out = subprocess.run(["ngspice", "-b", "maxpower.cir"], capture_output=True, text=True).stdout
    ng = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                ng[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    for k in ("ex1pmax", "ex2pmax", "ex4pmax", "p5pmax", "p12pmax"):
        print(f"{k:8s} {ng[k]:.6g} W")
    assert abs(ng["ex1pmax"] - 3.125) < 1e-6
else:
    print("ngspice not found; see SETUP.md. Skipping.")
p = lambda V, R, RL: V ** 2 * RL / (R + RL) ** 2
print(f"factor 2 off: {p(10, 8, 4):.4f} W and {p(10, 8, 16):.4f} W = {p(10, 8, 16) / 3.125 * 100:.1f}% of the maximum")

ex1pmax  3.125 W
ex2pmax  360 W
ex4pmax  0.03125 W
p5pmax   2304 W
p12pmax  10.7143 W
factor 2 off: 2.7778 W and 2.7778 W = 88.9% of the maximum


## Part 2 — One curve for every source

**🔮 Predict:** for random sources and loads, will $p/p_{max}$ plotted against efficiency fall on one curve?

In [3]:
rng = np.random.default_rng(609)
V = rng.uniform(1, 100, 300); R = 10 ** rng.uniform(-2, 3, 300); RL = R * 10 ** rng.uniform(-2, 2, 300)
eta = RL / (R + RL)
ratio = p(V, R, RL) / (V ** 2 / (4 * R))
e = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(eta, ratio, ".", ms=3, label="random sources and loads")
ax.plot(e, 4 * e * (1 - e), lw=1.2, label="4η(1 − η)")
ax.set_xlabel("efficiency η"); ax.set_ylabel("p / p_max"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
print("largest deviation from 4η(1−η):", np.abs(ratio - 4 * eta * (1 - eta)).max())
assert np.allclose(ratio, 4 * eta * (1 - eta))

largest deviation from 4η(1−η): 5.412337245047638e-16


Every point lies on $4\eta(1 - \eta)$, whatever the source's voltage and resistance: the tradeoff between power and efficiency is the same for every source that is a voltage behind a physical series resistance. (The sources simulated here are all of that form; for a general network, the Thévenin model does not give the internal loss, and the curve does not apply: chapter §3.2.)

## Part 3 — Tracking the maximum-power point

A **perturb-and-observe** tracker nudges its operating voltage, measures the power, and keeps going in whichever direction increased it. Apply it to the TEG of Example 06-09.4 (0.5 V behind 2 Ω) and to a simple solar-cell model, $i = I_L - I_0(e^{v/nV_T} - 1)$ with $I_L$ = 50 mA, $I_0$ = 1 nA, $n$ = 1.5.

**🔮 Predict:** will the tracker settle at half the open-circuit voltage for both sources?

In [4]:
VT = 0.025852
teg = lambda v: v * (0.5 - v) / 2.0
IL, I0, n = 50e-3, 1e-9, 1.5
solar = lambda v: v * (IL - I0 * np.expm1(v / (n * VT)))
voc_solar = n * VT * np.log(IL / I0 + 1)

def perturb_and_observe(power, v0, step, n_steps=1500):
    v, pv, direction, trace = v0, power(v0), 1.0, []
    for _ in range(n_steps):
        v_new = v + direction * step
        p_new = power(v_new)
        if p_new < pv:
            direction = -direction             # power fell: turn around
        v, pv = v_new, p_new
        trace.append(v)
    return np.array(trace)

for name, f, voc in (("TEG", teg, 0.5), ("solar cell", solar, voc_solar)):
    tr = perturb_and_observe(f, 0.05 * voc, 0.002 * voc)
    exact = minimize_scalar(lambda x: -f(x), bounds=(0, voc), method="bounded", options={"xatol": 1e-12})
    print(f"{name:10s}: Voc = {voc:.4f} V; tracker settles near {tr[-50:].mean():.4f} V; "
          f"true MPP {exact.x:.4f} V = {exact.x / voc * 100:.1f}% of Voc, P_mp = {-exact.fun * 1e3:.3f} mW")
    assert abs(tr[-50:].mean() - exact.x) < 2 * 0.002 * voc          # settled within two steps of the MPP
    if name == "TEG":
        assert abs(exact.x - 0.25) < 1e-6
    else:
        assert exact.x / voc > 0.75

TEG       : Voc = 0.5000 V; tracker settles near 0.2500 V; true MPP 0.2500 V = 50.0% of Voc, P_mp = 31.250 mW
solar cell: Voc = 0.6874 V; tracker settles near 0.5802 V; true MPP 0.5800 V = 84.4% of Voc, P_mp = 27.184 mW


The tracker finds the true maximum for both, oscillating by one step around it. For the TEG that is exactly $V_{oc}/2$; for the solar cell it is about 84% of $V_{oc}$, because a diode's current stays nearly constant until the voltage approaches $V_{oc}$ and then collapses. A tracker that assumed "half of $V_{oc}$" would collect only 63% of the solar cell's available power, losing 37%.

## Part 4 — Coax optima and the dielectric

**🔮 Predict:** as $\varepsilon_r$ rises from 1 (air) to 2.25 (polyethylene), do the optimal *ratios* $b/a$ change, or only the impedances?

In [5]:
f = lambda x: (1 + x) / np.log(x)
g = lambda x: np.log(x) / x ** 2
x_loss = minimize_scalar(f, bounds=(1.5, 10), method="bounded", options={"xatol": 1e-12}).x
x_pow = minimize_scalar(lambda x: -g(x), bounds=(1.1, 5), method="bounded", options={"xatol": 1e-12}).x
print(f"optimal b/a: loss {x_loss:.4f}, power {x_pow:.4f} (independent of the dielectric)")
for er in (1.0, 1.4, 2.1, 2.25):
    z_loss = 60 / np.sqrt(er) * np.log(x_loss)
    z_pow = 60 / np.sqrt(er) * np.log(x_pow)
    x50 = np.exp(50 * np.sqrt(er) / 60)
    print(f"eps_r = {er:4.2f}: min-loss Z0 = {z_loss:6.2f} ohm, max-power Z0 = {z_pow:6.2f} ohm; "
          f"a 50 ohm line has {100 * (f(x50) / f(x_loss) - 1):5.2f}% extra loss, {100 * g(x50) / g(x_pow):5.1f}% of max power")
assert abs(x_pow - np.sqrt(np.e)) < 1e-6 and abs(60 * np.log(x_loss) - 76.708) < 1e-3

optimal b/a: loss 3.5911, power 1.6487 (independent of the dielectric)
eps_r = 1.00: min-loss Z0 =  76.71 ohm, max-power Z0 =  30.00 ohm; a 50 ohm line has 10.30% extra loss,  85.6% of max power
eps_r = 1.40: min-loss Z0 =  64.83 ohm, max-power Z0 =  25.35 ohm; a 50 ohm line has  3.94% extra loss,  74.6% of max power
eps_r = 2.10: min-loss Z0 =  52.93 ohm, max-power Z0 =  20.70 ohm; a 50 ohm line has  0.20% extra loss,  58.7% of max power
eps_r = 2.25: min-loss Z0 =  51.14 ohm, max-power Z0 =  20.00 ohm; a 50 ohm line has  0.03% extra loss,  55.8% of max power


The optimal ratios depend only on geometry; the dielectric scales every impedance by $1/\sqrt{\varepsilon_r}$. With a solid dielectric ($\varepsilon_r \approx 2$), 50 Ω lands almost exactly on the minimum-loss design, while the maximum-power impedance drops to about 20 Ω.

## Your turn

1. In Part 3, what fraction of the solar cell's maximum power would a tracker that held $V_{oc}/2$ collect?
2. In Part 2, what efficiency gives exactly half of the maximum power (two answers)?
3. In Part 4, foam dielectrics have $\varepsilon_r$ near 1.4. Which impedance minimizes loss for them?

<details><summary>Answers</summary>

1. At $v = V_{oc}/2$ the diode current is negligible, so $i \approx I_L$ and $p \approx (V_{oc}/2)I_L = 0.344\times50\ \text{mA} = 17.2$ mW: 63% of the 27.2 mW maximum.
2. $4\eta(1 - \eta) = 1/2$ gives $\eta = (1 \pm 1/\sqrt2)/2$: **14.6%** or **85.4%**. Power systems would choose the 85.4% point if they had to deliver half the maximum.
3. $76.7/\sqrt{1.4} = 64.8$ Ω (the notebook prints it).

</details>